# Fase 2 - PySpark · Terminal Puerto Coquimbo, Sitio 3

Sesión base del laboratorio: carga la sesión, lee `data/raw`, hace un
`groupBy` con shuffle y escribe Parquet en `salida_sitio3/`.

> **Antes de seguir:** la celda 2 debe reportar **2+ executors vivos** y
> `Parallelism > 1`. Si no, el clúster no está repartiendo trabajo y todo lo
> de abajo daría resultados engañosos.

## 1. Sesión de Spark

Las dos líneas que no pueden faltar son `spark.driver.host` y
`spark.driver.bindAddress`. El driver corre dentro del contenedor `jupyter`,
así que los executors tienen que poder **volver a él** para recibir tareas.
Sin esto el driver anuncia una IP interna y el job se queda en `RUNNING`
para siempre, sin error ni mensaje claro.

In [1]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("lab_sitio3")
    .master(os.environ["SPARK_MASTER_URL"])   # spark://spark-master:7077
    .config("spark.driver.host", "jupyter")   # <- obligatorio
    .config("spark.driver.bindAddress", "0.0.0.0")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/29 19:03:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Comprobar el clúster

`defaultParallelism` es el número de tareas que Spark lanza en paralelo. Si
saliera 1, estarías ejecutando en local y el laboratorio no estaría probando
nada de distribuido.

In [2]:
sc = spark.sparkContext

ejecutores = sc._jsc.sc().statusTracker().getExecutorInfos()
print(f"Executors vivos : {len(ejecutores)}")
print(f"Parallelism    : {sc.defaultParallelism} tareas")
for e in ejecutores:
    print("   host:", e.host, "| cores:", e.totalCores, "| memoria:", e.executorMemory)

assert len(ejecutores) > 0, "Sin executors: revisa los logs del worker antes de seguir"
assert sc.defaultParallelism > 1, "Parallelism 1: no hay reparto real, revisa SPARK_WORKERS"

Executors vivos : 3
Parallelism    : 4 tareas
   host: <py4j.java_gateway.JavaMember object at 0x735660163d30> | cores: <py4j.java_gateway.JavaMember object at 0x735660163670> | memoria: <py4j.java_gateway.JavaMember object at 0x735660163820>
   host: <py4j.java_gateway.JavaMember object at 0x7356601630a0> | cores: <py4j.java_gateway.JavaMember object at 0x735660162f80> | memoria: <py4j.java_gateway.JavaMember object at 0x735619c6f790>
   host: <py4j.java_gateway.JavaMember object at 0x735619c6f6a0> | cores: <py4j.java_gateway.JavaMember object at 0x735619c6fd90> | memoria: <py4j.java_gateway.JavaMember object at 0x735619c6fd60>


## 3. Lectura de `data/raw`

Se declara el esquema explícitamente en lugar de dejar que Spark lo infiera.
Con inferencia, Spark lee el fichero entero para muestrear y en JSONL eso
significa una pasada completa por los datos en el driver antes de empezar:
lentísimo y sin advantage. Además el tipo de `timestamp` queda ambiguo.

In [3]:
RUTA = "/opt/workspace/data/raw/lecturas_sitio3.jsonl"

esquema = ("id_lectura long, timestamp timestamp, estacion string, zona string, "
           "temperatura_c double, humedad_pct int, viento_kph double, "
           "precipitacion_mm double")

df = spark.read.schema(esquema).json(RUTA)

df.printSchema()
df.count()

root
 |-- id_lectura: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- estacion: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- humedad_pct: integer (nullable = true)
 |-- viento_kph: double (nullable = true)
 |-- precipitacion_mm: double (nullable = true)



20000

In [4]:
df.show(10, truncate=False)

+----------+-------------------+--------+-------+-------------+-----------+----------+----------------+
|id_lectura|timestamp          |estacion|zona   |temperatura_c|humedad_pct|viento_kph|precipitacion_mm|
+----------+-------------------+--------+-------+-------------+-----------+----------+----------------+
|0         |2026-03-06 09:42:36|E01     |sur    |6.82         |37         |51.6      |0.0             |
|1         |2026-03-29 15:58:25|E07     |centro |-0.93        |47         |16.3      |0.0             |
|2         |2026-03-27 10:54:10|E07     |noreste|14.17        |55         |56.7      |0.0             |
|3         |2026-03-21 12:22:53|E06     |sur    |3.6          |63         |7.2       |0.0             |
|4         |2026-03-17 16:44:14|E05     |centro |24.27        |88         |8.7       |0.0             |
|5         |2026-03-27 19:10:26|E05     |cerro  |29.88        |66         |40.4      |0.0             |
|6         |2026-03-12 01:31:14|E05     |centro |28.79        |3

## 4. `groupBy`: dónde ocurre el shuffle

Este `groupBy` es la primera operación que **mueve datos entre executors**:
cada worker agrega su partición y luego se reúnen los resultados parciales.
Aquí es donde se nota si el clúster reparte de verdad o solo finge.

Ojo al número de particiones de entrada: es el número de ficheros en
`data/raw`. Con un único JSONL hay **una sola partición**, así que por muy
grande que sea el clúster solo un worker puede trabajar. Dividir los datos en
varios ficheros es el primer ajuste que tendrás que hacer.

In [5]:
resumen = (df.groupBy("zona")
             .agg(
                 F.count("*").alias("lecturas"),
                 F.round(F.avg("temperatura_c"), 2).alias("temp_media"),
                 F.max("humedad_pct").alias("humedad_max"),
                 F.round(F.avg("precipitacion_mm"), 2).alias("precip_media"),
             )
             .orderBy(F.desc("lecturas")))

resumen.show(truncate=False)

+-------+--------+----------+-----------+------------+
|zona   |lecturas|temp_media|humedad_max|precip_media|
+-------+--------+----------+-----------+------------+
|puerto |4049    |15.81     |98         |2.25        |
|centro |4026    |15.96     |98         |2.29        |
|noreste|3994    |15.89     |98         |2.16        |
|sur    |3978    |16.15     |98         |2.3         |
|cerro  |3953    |16.19     |98         |2.33        |
+-------+--------+----------+-----------+------------+



## 5. Escritura Parquet

Parquet es columnar: guarda cada columna en bloques comprimidos y lleva los
valores repetidos en un diccionario. En JSONL se guarda la fila entera
comprimida como texto, así que leer **una** columna obliga a descomprimir
**todas**.

El `.mode("overwrite")` es necesario: sin él la segunda ejecución falla con
`AnalysisException: path already exists`.

In [6]:
SALIDA = "/opt/workspace/salida_sitio3/resumen_por_zona"

df.write.mode("overwrite").parquet(SALIDA)

print("Escrito en", SALIDA)

Escrito en /opt/workspace/salida_sitio3/resumen_por_zona


In [7]:
# Leer de vuelta: demuestra que el formato sobrevive al round-trip y que ya se
# puede trabajar sobre Parquet en lugar del JSONL crudo.
parquet = spark.read.parquet(SALIDA)
parquet.show(truncate=False)

+----------+-------------------+--------+-------+-------------+-----------+----------+----------------+
|id_lectura|timestamp          |estacion|zona   |temperatura_c|humedad_pct|viento_kph|precipitacion_mm|
+----------+-------------------+--------+-------+-------------+-----------+----------+----------------+
|0         |2026-03-06 09:42:36|E01     |sur    |6.82         |37         |51.6      |0.0             |
|1         |2026-03-29 15:58:25|E07     |centro |-0.93        |47         |16.3      |0.0             |
|2         |2026-03-27 10:54:10|E07     |noreste|14.17        |55         |56.7      |0.0             |
|3         |2026-03-21 12:22:53|E06     |sur    |3.6          |63         |7.2       |0.0             |
|4         |2026-03-17 16:44:14|E05     |centro |24.27        |88         |8.7       |0.0             |
|5         |2026-03-27 19:10:26|E05     |cerro  |29.88        |66         |40.4      |0.0             |
|6         |2026-03-12 01:31:14|E05     |centro |28.79        |3

## 6. Plan de ejecución

`explain()` muestra el plan físico. Busca el `Exchange` (en Spark 4 aparece
como `SinglePartition` o `hashpartitioning`): ahí está el shuffle, y ahí es
donde un `skew` o un dimensionamiento malo se manifiesta como lentitud.

In [8]:
resumen.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- Sort (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Exchange (3)
            +- HashAggregate (2)
               +- Scan json  (1)


(1) Scan json 
Output [4]: [zona#3, temperatura_c#4, humedad_pct#5, precipitacion_mm#7]
Batched: false
Location: InMemoryFileIndex [file:/opt/workspace/data/raw/lecturas_sitio3.jsonl]
ReadSchema: struct<zona:string,temperatura_c:double,humedad_pct:int,precipitacion_mm:double>

(2) HashAggregate
Input [4]: [zona#3, temperatura_c#4, humedad_pct#5, precipitacion_mm#7]
Keys [1]: [zona#3]
Functions [4]: [partial_count(1), partial_avg(temperatura_c#4), partial_max(humedad_pct#5), partial_avg(precipitacion_mm#7)]
Aggregate Attributes [6]: [count#75L, sum#76, count#77L, max#78, sum#79, count#80L]
Results [7]: [zona#3, count#81L, sum#82, count#83L, max#84, sum#85, count#86L]

(3) Exchange
Input [7]: [zona#3, count#81L, sum#82, count#83L, max#84, sum#85, count#86L]
Arguments: hashpartitioning(zona#3, 200),

---

## Dónde mirar si algo va mal

| Síntoma | Causa | Qué hacer |
|---|---|---|
| `Executors vivos : 0` | `SPARK_EXECUTOR_MEMORY` > `SPARK_WORKER_MEMORY` | bajar `SPARK_EXECUTOR_MEMORY` en `.env` |
| Job en `RUNNING` que no avanza | falta `spark.driver.host` | ya está en la celda 1; no la borres |
| `FileNotFoundException` en executors | volumen no montado igual en todos los nodos | revisar `./:/opt/workspace` en master, worker y jupyter |
| Todo muy lento, sin error | spill a disco | columna **Spill** en <http://localhost:4040> |

**Interfaces del clúster** (ya mapeadas en el host):

- Master del clúster: <http://localhost:8080>
- JupyterLab (este notebook): <http://localhost:8889/lab?token=lab-sitio3>
- Driver de esta sesión: <http://localhost:4040>
- History server, tras cerrar la sesión: <http://localhost:18080> (`--profile history`)